# ScholarRAG: M5, answer evaluation

M4 showed that matching cited chunks against gold chunks undercounts correct answers, because the same fact often appears in several chunks of a paper. This notebook evaluates the M4 answers directly.

| Part | What | How |
|---|---|---|
| 1 | Out-of-corpus refusal (set C) | Run the M4 pipeline on 12 LLM-generated questions about papers **not** in the corpus |
| 2 | Answer correctness | LLM judge (`gemini-3.5-flash`, a stronger model than the one that answered) compares each answer with the reference answer and the cited passages |
| 3 | Judge check | A blind manual sample: you label it, the notebook measures agreement with the judge |
| 4 | Refusal accuracy | Sets B2 (near-miss) and C (out of corpus), plus false refusals on answerable questions |
| 5 | Cost and latency | Tokens per question and seconds per stage (retrieval, rerank, LLM) |

**Judge verdicts**

| Verdict | Meaning |
|---|---|
| `correct` | Gives the key information of the reference answer (wording may differ; extra correct detail is fine) |
| `correct_different` | Differs from the reference but answers the question, and every claim is supported by the cited passages |
| `partial` | Answers only part of the question, or gives a different detail from the same paper instead of the one asked for |
| `incorrect` | Contradicts the reference and is not supported by the cited passages, or does not answer the question |

`correct` + `correct_different` count as **acceptable**. Separately, `supported` says whether every claim in the answer is stated in the cited passages.

The pipeline code lives in `src/rag.py` (same prompt and settings as notebooks 03–04). All LLM outputs are cached in `eval/m5_*.jsonl`, so a re-run does not call the APIs again.

## 0. Setup

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd
from pathlib import Path
from typing import Literal
from dotenv import load_dotenv

ROOT = Path.home() / "Projects" / "scholar-rag"
sys.path.insert(0, str(ROOT / "src"))
import rag
BASE = rag.BASE
load_dotenv(ROOT / ".env")

JUDGE_MODEL = "gemini-3.5-flash"
JUDGE_VERSION = "j1"
ANSWER_SLEEP, JUDGE_SLEEP = 4.5, 6.5      # jeda antar panggilan (batas free tier)
N_LATENCY = 20                            # soal M4 yang diukur ulang waktu retrieval-nya
N_MANUAL = 25                             # ukuran sampel manual
# harga per 1 juta token (USD); isi dari halaman harga Gemini untuk menghitung biaya
PRICE = {"gemini-3.5-flash-lite": {"in": None, "out": None}, "gemini-3.5-flash": {"in": None, "out": None}}

from google import genai
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
ch = rag.load_chunks()
pd.set_option("display.max_colwidth", 160)

## 1. Load the M4 results

In [ ]:
m4 = pd.read_csv(BASE / "eval" / "m4_results.csv", dtype={"paper_id": str})
m4_raw = {json.loads(l)["id"]: json.loads(l) for l in open(BASE / "eval" / "m4_answers.jsonl")}
pos = {c: i for i, c in enumerate(ch.chunk_id)}

items = []    # satu baris per soal: set, soal, jawaban referensi, keluaran sistem
names = {"llm": "A", "handwritten": "B1", "unanswerable": "B2"}
for r in m4.itertuples():
    a = m4_raw[r.id]
    items.append({"id": r.id, "set": names[r.source], "answerable": bool(r.answerable), "question": r.question,
                  "reference": r.gold_answer if isinstance(r.gold_answer, str) else "",
                  "parsed": a["parsed"], "idxs": [pos[c] for c in a["context_chunk_ids"]],
                  "gold_in_ctx": bool(r.gold_in_ctx), "latency_s": a["latency_s"],
                  "tokens_in": a["tokens_in"], "tokens_out": a["tokens_out"]})
print(pd.Series([i["set"] for i in items]).value_counts().to_dict())

## 2. Set C: questions about papers outside the corpus

`eval/unanswerable_candidates.csv` holds 12 LLM-generated questions whose source papers are not among the 200 corpus papers. They were never reviewed by hand, so they are reported separately. The correct behaviour is to refuse, **unless** a corpus paper genuinely states the answer (some are well-known methods that corpus papers may describe); the judge in section 3 separates those cases.

This part also times the retrieval stages on 20 M4 questions, for the latency report.

In [ ]:
setc_path, lat_path = BASE / "eval" / "m5_setC.jsonl", BASE / "eval" / "m5_latency.csv"
uc = pd.read_csv(BASE / "eval" / "unanswerable_candidates.csv", dtype=str).fillna("")
setc = {json.loads(l)["id"]: json.loads(l) for l in open(setc_path)} if setc_path.exists() else {}
need_c = [r for r in uc.itertuples() if r.id not in setc]
need_lat = not lat_path.exists()

if need_c or need_lat:
    R = rag.Retriever(ch)
    with open(setc_path, "a") as f:
        for r in need_c:
            idxs, scores, t = R.retrieve(r.question)
            a = rag.answer(client, ch, r.question, idxs)
            rec = {"id": r.id, "chunk_ids": ch.chunk_id.iloc[idxs].tolist(), "rerank_scores": scores, "retrieval_s": t, **a}
            setc[r.id] = rec
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            time.sleep(ANSWER_SLEEP)
    if need_lat:
        rows = []
        sample = m4.sample(N_LATENCY, random_state=0)
        for q in sample.itertuples():
            idxs, _, t = R.retrieve(q.question)
            same = ch.chunk_id.iloc[idxs].tolist() == m4_raw[q.id]["context_chunk_ids"]
            rows.append({"id": q.id, **{f"{k}_s": v for k, v in t.items()}, "same_top5_as_m4": same})
        pd.DataFrame(rows).to_csv(lat_path, index=False)
    del R

for r in uc.itertuples():
    a = setc[r.id]
    items.append({"id": r.id, "set": "C", "answerable": False, "question": r.question, "reference": r.answer,
                  "parsed": a["parsed"], "idxs": [pos[c] for c in a["chunk_ids"]], "gold_in_ctx": False,
                  "latency_s": a["latency_s"], "tokens_in": a["tokens_in"], "tokens_out": a["tokens_out"]})
lat = pd.read_csv(lat_path)
print(len(need_c), "soal set C dijawab sekarang |", int(lat.same_top5_as_m4.sum()), f"/ {len(lat)} top-5 sama dengan M4")

## 3. LLM judge

Every **answered** question in sets A, B1 and C is judged. The judge sees the question, the reference answer, the system's answer, and the full text of the passages the system cited.

In [ ]:
from google.genai import types
from pydantic import BaseModel, Field

class Judgment(BaseModel):
    verdict: Literal["correct", "correct_different", "partial", "incorrect"]
    supported: bool = Field(description="True only if every factual claim in the answer is stated in the cited passages")
    reason: str = Field(description="One sentence")

JUDGE_SYSTEM = """You grade answers from a question-answering system over research papers on visual anomaly detection.

You receive the question, a reference answer written from one passage of the source paper, the system's answer, and the passages the system cited.

Verdict:
- correct: the answer gives the key information of the reference answer (wording may differ; extra correct detail is fine).
- correct_different: the answer does not match the reference, but it answers the question and every claim is supported by the cited passages (the reference may be too narrow, or the paper states it differently elsewhere).
- partial: the answer covers only part of what the question asks, or gives a different detail from the same paper instead of the specific item requested.
- incorrect: the answer contradicts the reference and is not supported by the cited passages, or does not answer the question.

supported: true only if every factual claim in the answer is stated in the cited passages.

Judge only from the texts given."""

def judge_prompt(it):
    p = it["parsed"]
    cited = sorted({c["passage"] for c in p["citations"] if 1 <= c["passage"] <= len(it["idxs"])})
    passages = "\n\n".join(f"[{n}] {ch.text.iat[it['idxs'][n - 1]]}" for n in cited) or "(no valid citation)"
    quotes = "\n".join(f"- [{c['passage']}] {c['quote']}" for c in p["citations"]) or "(none)"
    return (f"Question: {it['question']}\n\nReference answer: {it['reference']}\n\n"
            f"System answer: {p['answer']}\n\nQuotes the system cited:\n{quotes}\n\nCited passages:\n\n{passages}")

judge_cfg = types.GenerateContentConfig(system_instruction=JUDGE_SYSTEM, temperature=0,
                                        response_mime_type="application/json", response_schema=Judgment)
judg_path = BASE / "eval" / "m5_judgments.jsonl"
judgments = {}
if judg_path.exists():
    for l in open(judg_path):
        r = json.loads(l)
        if r["judge_version"] == JUDGE_VERSION and r["model"] == JUDGE_MODEL and r["parsed"] is not None:
            judgments[r["id"]] = r

to_judge = [it for it in items if it["set"] != "B2" and it["parsed"] and it["parsed"]["answerable"] and it["id"] not in judgments]
with open(judg_path, "a") as f:
    for it in to_judge:
        t0 = time.perf_counter()
        try:
            r = rag.call_with_retry(lambda: client.models.generate_content(model=JUDGE_MODEL, contents=judge_prompt(it), config=judge_cfg))
        except rag.DailyQuotaExceeded:
            print("kuota harian model judge habis; jalankan ulang sel ini besok, penilaian yang sudah ada tetap tersimpan")
            break
        u = r.usage_metadata
        rec = {"id": it["id"], "judge_version": JUDGE_VERSION, "model": JUDGE_MODEL,
               "parsed": r.parsed.model_dump() if r.parsed is not None else None,
               "latency_s": round(time.perf_counter() - t0, 2), "tokens_in": u.prompt_token_count,
               "tokens_out": (u.candidates_token_count or 0) + (getattr(u, "thoughts_token_count", 0) or 0)}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
        if rec["parsed"] is not None:
            judgments[it["id"]] = rec
        time.sleep(JUDGE_SLEEP)
print(len(to_judge), "dinilai sekarang |", len(judgments), "penilaian tersimpan")

In [ ]:
rows = []
for it in items:
    p, j = it["parsed"], judgments.get(it["id"])
    answered = bool(p and p["answerable"])
    rows.append({"id": it["id"], "set": it["set"], "answerable": it["answerable"], "answered": answered,
                 "gold_in_ctx": it["gold_in_ctx"],
                 "citations_valid": rag.check_citations(ch, p, it["idxs"]) if answered else np.nan,
                 "verdict": j["parsed"]["verdict"] if (answered and j) else ("refused" if not answered else "not judged"),
                 "supported": j["parsed"]["supported"] if (answered and j) else np.nan,
                 "reason": j["parsed"]["reason"] if (answered and j) else "",
                 "question": it["question"], "reference": it["reference"], "answer": p["answer"] if p else ""})
ev = pd.DataFrame(rows)
ev.to_csv(BASE / "eval" / "m5_results.csv", index=False)

VERDICTS = ["correct", "correct_different", "partial", "incorrect", "refused"]
quality = (pd.crosstab(ev[ev.answerable].set, ev[ev.answerable].verdict)
             .reindex(index=["A", "B1"], columns=VERDICTS + ["not judged"], fill_value=0))
if quality["not judged"].sum() == 0:
    quality = quality.drop(columns="not judged")
quality["n"] = quality.sum(axis=1)
quality["acceptable"] = ((quality.correct + quality.correct_different) / quality.n).round(3)
ans = ev[ev.answerable & ev.answered & ev.supported.notna()]
quality["supported (of answered)"] = ans.groupby("set").supported.apply(lambda s: s.astype(float).mean()).reindex(quality.index).round(3)
quality

### 3.1 Answers the judge did not accept

In [ ]:
ev[ev.answerable & ev.verdict.isin(["partial", "incorrect"])][["id", "set", "verdict", "supported", "question", "reference", "answer", "reason"]]

### 3.2 Accepted, but different from the reference

These are the cases where the reference answer may be too narrow. Worth a look before the next version of the question sets.

In [ ]:
ev[ev.verdict == "correct_different"][["id", "set", "question", "reference", "answer", "reason"]]

## 4. Refusal accuracy

- **B2 (near-miss):** the paper is in the corpus, the detail is not. Refusing is correct.
- **C (out of corpus):** refusing is correct; an answer is acceptable only if the judge finds it fully supported by the cited corpus passages.
- **False refusals:** answerable questions (A + B1) that were refused, split by whether the gold chunk was retrieved.

In [ ]:
def refusal_row(d, name):
    answered = d[d.answered]
    return {"set": name, "n": len(d), "refused": int((~d.answered).sum()),
            "answered, supported": int((answered.supported == True).sum()),
            "answered, not supported": int((answered.supported == False).sum()),
            "refusal rate": round((~d.answered).mean(), 3)}

ref = pd.DataFrame([refusal_row(ev[ev.set == "B2"], "B2 near-miss"), refusal_row(ev[ev.set == "C"], "C out of corpus")]).set_index("set")
a = ev[ev.answerable]
false_ref = pd.Series({"answerable questions": len(a), "refused": int((~a.answered).sum()),
                       "refused with gold retrieved (LLM error)": int((~a.answered & a.gold_in_ctx).sum()),
                       "refused, gold not retrieved (retrieval miss)": int((~a.answered & ~a.gold_in_ctx).sum())})
display(ref)
false_ref

In [ ]:
# set C yang dijawab: apa yang dijawab dan apakah didukung passage korpus
ev[(ev.set == "C") & ev.answered][["id", "verdict", "supported", "question", "answer", "reason"]]

## 5. Check the judge against your own labels

The judge is itself an LLM, so its verdicts need a check. This cell writes a **blind** sample to `eval/m5_manual_sample.csv`: every answer the judge did not rate `correct`, plus random `correct` ones, up to 25 rows. The judge's verdicts go to a separate file (`eval/m5_manual_key.csv`) so they do not influence your labels.

**To label:** open `m5_manual_sample.csv`, fill `manual_verdict` (one of `correct`, `correct_different`, `partial`, `incorrect`) and `manual_supported` (`TRUE`/`FALSE`) for each row, save, and re-run this section. The files are never overwritten once they exist.

In [ ]:
sample_path, key_path = BASE / "eval" / "m5_manual_sample.csv", BASE / "eval" / "m5_manual_key.csv"
judged = ev[ev.verdict.isin(VERDICTS[:4])]
if not sample_path.exists():
    hard = judged[judged.verdict != "correct"]
    easy = judged[judged.verdict == "correct"].sample(max(0, N_MANUAL - len(hard)), random_state=0)
    sample = pd.concat([hard, easy]).sample(frac=1, random_state=1)        # acak urutan supaya tetap buta
    by_id = {it["id"]: it for it in items}
    def quotes(i):
        return " | ".join(f"[{c['passage']}] {c['quote']}" for c in by_id[i]["parsed"]["citations"])
    out = sample[["id", "set", "question", "reference", "answer"]].copy()
    out["cited_quotes"] = out.id.map(quotes)
    out["manual_verdict"], out["manual_supported"], out["manual_note"] = "", "", ""
    out.to_csv(sample_path, index=False)
    sample[["id", "verdict", "supported", "reason"]].to_csv(key_path, index=False)
    print(f"sampel manual dibuat: {len(out)} baris ({len(hard)} bukan 'correct', {len(easy)} 'correct')")
else:
    print("sampel manual sudah ada, tidak ditimpa")

In [ ]:
man = pd.read_csv(sample_path, dtype=str).fillna("").merge(pd.read_csv(key_path, dtype=str), on="id")
done = man[man.manual_verdict.str.strip() != ""]
if done.empty:
    print(f"belum ada label manual (0 / {len(man)}). Isi manual_verdict di {sample_path.name}, lalu jalankan ulang sel ini.")
else:
    acc = lambda v: v.isin(["correct", "correct_different"])
    print(f"{len(done)} / {len(man)} berlabel")
    print("verdict sama persis       :", round((done.manual_verdict == done.verdict).mean(), 3))
    print("acceptable vs tidak sama  :", round((acc(done.manual_verdict) == acc(done.verdict)).mean(), 3))
    lab = done[done.manual_supported.str.strip() != ""]
    if len(lab):
        print("supported sama            :", round((lab.manual_supported.str.upper() == lab.supported.str.upper()).mean(), 3))
    display(pd.crosstab(done.manual_verdict.rename("manual"), done.verdict.rename("judge")))
    display(done[done.manual_verdict != done.verdict][["id", "manual_verdict", "verdict", "manual_note", "reason"]])

## 6. Cost and latency

Tokens come from the API's usage metadata (the judge's count includes its thinking tokens). Set the prices in `PRICE` (section 0) to turn tokens into dollars.

In [ ]:
ans_tok = pd.DataFrame([{"set": it["set"], "in": it["tokens_in"], "out": it["tokens_out"], "llm_s": it["latency_s"]} for it in items])
jud_tok = pd.DataFrame([{"in": j["tokens_in"], "out": j["tokens_out"], "s": j["latency_s"]} for j in judgments.values()])

def cost(model, tin, tout):
    p = PRICE[model]
    return None if p["in"] is None else round(tin / 1e6 * p["in"] + tout / 1e6 * p["out"], 4)

tokens = pd.DataFrame({
    "calls": [len(ans_tok), len(jud_tok)],
    "tokens in / call": [round(ans_tok["in"].mean()), round(jud_tok["in"].mean())],
    "tokens out / call": [round(ans_tok["out"].mean()), round(jud_tok["out"].mean())],
    "total cost USD": [cost(rag.ANSWER_MODEL, ans_tok["in"].sum(), ans_tok["out"].sum()),
                       cost(JUDGE_MODEL, jud_tok["in"].sum(), jud_tok["out"].sum())],
}, index=[f"answer ({rag.ANSWER_MODEL})", f"judge ({JUDGE_MODEL})"])
display(tokens)

stages = lat[["dense_s", "bm25_s", "rerank_s"]].copy()
stages["retrieval_total_s"] = stages.sum(axis=1)
llm = ans_tok.llm_s
end_to_end = stages.retrieval_total_s.median() + llm
latency = pd.DataFrame({
    "median s": [stages.dense_s.median(), stages.bm25_s.median(), stages.rerank_s.median(), llm.median(), end_to_end.median()],
    "p90 s": [stages.dense_s.quantile(.9), stages.bm25_s.quantile(.9), stages.rerank_s.quantile(.9), llm.quantile(.9), end_to_end.quantile(.9)],
}, index=["dense search", "BM25", "rerank (≈16 pairs, M1)", "LLM answer", "end to end (est.)"]).round(2)
latency

## 7. Reading the results

_To be written after the run._